# Classify Intents with Encoders

Requests to a voice assistant ("wake me up at nine", "play some jazz"), each with one of 60 intents: a real classification problem, with many classes, imbalanced and sometimes close. The data: Amazon's [MASSIVE](https://github.com/alexa/massive).

1. Classic models on TF-IDF: the baseline, then character n-grams
2. Frozen *embeddings* + logistic regression
3. Fine-tune an encoder
4. Keep the results for tomorrow (the next lab compares with an LLM), then read the F1 closely
5. Bonus: SetFit with 8 examples per class, and calibration

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs: the models of exercises 2, 3 and 5, too slow for a CPU, see a sample only there, then the next exercise loads their result computed on a GPU.

## Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh massive-intents

In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_parquet("massive-intents/en-train.parquet")
test = pd.read_parquet("massive-intents/en-test.parquet")
labels = sorted(train.label.unique())
GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# On a CPU, float32: half precisions (float16, and bfloat16, the published weights' one,
# which we'd get by default) are very slow there.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "No GPU: the long computations will load their result")
# What the models of exercises 2, 3 and 5 see: everything on a GPU; without one, the first
# 64 training and test texts.
n = None if GPU else 64
train_part, test_part = train[:n], test[:n]
print(len(train), "training examples,", len(test), "test,", len(labels), "intents")
train.label.value_counts().describe()

`record` measures and keeps each method's results: the macro F1 (every intent of the test counts the same), the accuracy, the training time and the prediction time per text. `confusions` shows the pairs of intents most often confused.

In [ ]:
results = {}


def record(
  name: str, predictions, train_seconds: float, predict_seconds: float
) -> None:
  # Without a GPU, a method may have predicted the first test texts only.
  truth = test.label[: len(predictions)]
  # labels: the macro F1 covers the test's intents only. By default, an intent absent
  # from the test (cooking_query, in French) but predicted once would count, with an
  # F1 of 0.
  results[name] = {
    "predictions": list(predictions),
    "train_s": train_seconds,
    "ms_per_text": 1000 * predict_seconds / len(predictions),
  }
  print(
    f"{name:30} macro F1 {f1_score(truth, predictions, average='macro', labels=truth.unique()):.3f}"
    f"  accuracy {accuracy_score(truth, predictions):.3f}"
    f"  training {train_seconds:5.0f} s  {results[name]['ms_per_text']:.2f} ms/text"
  )


def confusions(predictions, n: int = 10) -> list[tuple[tuple[str, str], int]]:
  errors = Counter((t, p) for t, p in zip(test.label, predictions) if t != p)
  return errors.most_common(n)

## 1. The Baseline: Classic Models on TF-IDF

The official program's models, on a TF-IDF representation (words and word pairs). Run the cell (two to three minutes) and read the results: which one is best? Which intents does it confuse?

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train.text)
X_test = vectorizer.transform(test.text)
for name, model in (
  ("TF-IDF + Naive Bayes", MultinomialNB(alpha=0.1)),
  ("TF-IDF + linear SVM", LinearSVC()),
  ("TF-IDF + logistic regression", LogisticRegression(max_iter=2000, C=10)),
  (
    "TF-IDF + random forest",
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
  ),
  (
    "TF-IDF + LightGBM",
    LGBMClassifier(learning_rate=0.05, num_leaves=15, verbose=-1),
  ),
):
  start = time.time()
  model.fit(X_train, train.label)
  trained = time.time() - start
  start = time.time()
  predictions = model.predict(X_test)
  record(name, predictions, trained, time.time() - start)

# Trees (forests, gradient boosting), the kings of tabular data, do poorly on
# thousands of sparse columns. LightGBM (gradient boosting) even diverges with its
# default settings (an F1 close to 0); with a smaller step and smaller trees
# (learning_rate=0.05, num_leaves=15), it learns, but stays below the forest. On text,
# the classic baseline is a linear model on TF-IDF.

In [ ]:
confusions(results["TF-IDF + linear SVM"]["predictions"])

### Character N-Grams

These requests are short, spoken, and not always well spelled. For the linear SVM, replace the words with n-grams of 2 to 5 characters, taken inside the words, keeping the other settings of `TfidfVectorizer`, and record the result with `record`. Compare with the SVM on words: what do we gain, and why on these texts?

**Useful documentation**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)`: `"char_wb"` splits each word, padded with a space on each side, into character n-grams; `ngram_range=(2, 5)`: from 2 to 5 characters
- `.fit_transform(train.text)` on the training set, `.transform(test.text)` on the test set, as for words
- `LinearSVC().fit(X, train.label)`, then `.predict(X_test)`, timed as in the loop

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
char_vectorizer = TfidfVectorizer(
  analyzer=..., ngram_range=..., sublinear_tf=True, min_df=2
)
X_train_chars = char_vectorizer.fit_transform(train.text)
X_test_chars = char_vectorizer.transform(...)
start = time.time()
char_svm = LinearSVC().fit(..., train.label)
trained = time.time() - start
start = time.time()
predictions = char_svm.predict(...)
record("Character TF-IDF + SVM", predictions, trained, time.time() - start)
```

#### Solution

In [ ]:
char_vectorizer = TfidfVectorizer(
  analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2
)
X_train_chars = char_vectorizer.fit_transform(train.text)
X_test_chars = char_vectorizer.transform(test.text)
start = time.time()
char_svm = LinearSVC().fit(X_train_chars, train.label)
trained = time.time() - start
start = time.time()
predictions = char_svm.predict(X_test_chars)
record("Character TF-IDF + SVM", predictions, trained, time.time() - start)

# On words, macro F1 0.817 and accuracy 0.833; on characters, 0.808 and 0.839: about
# the same. A word never seen in training often has known pieces: "dimmer" isn't
# there, but "dim" is. Character n-grams thus bring the forms of a word together, and
# resist typos. English inflects little, and these texts have few typos: in French,
# the same change gains 2 points of macro F1.

## 2. Frozen *Embeddings* + Logistic Regression

Encode the training and test texts with Qwen3-Embedding-0.6B (`embedder`), each text preceded by the instruction below, which describes the task to the model. Train a logistic regression on the vectors, and record the result with `record` (the prediction time includes encoding the test texts). Work on `train_part` and `test_part`: the whole set on a GPU; without one, 64 texts of each (about a minute), and the next exercise loads the vectors of all of them, computed on a GPU.

**Useful documentation**

- [`SentenceTransformer.encode`](https://sbert.net/docs/package_reference/sentence_transformer/model.html#sentence_transformers.sentence_transformer.model.SentenceTransformer.encode)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
embedder.max_seq_length = 64
INSTRUCTION = "Instruct: Classify the intent of a request to a voice assistant\nQuery: "

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `embedder.encode(texts, prompt=INSTRUCTION, batch_size=64)`, on `train_part.text.tolist()` then `test_part.text.tolist()`
- `LogisticRegression(max_iter=2000, C=10).fit(vectors, train_part.label)`, then `.predict(test_vectors)`
- `record(name, predictions, training_seconds, prediction_seconds)`: time it with `time.time()`; training includes encoding the training texts, prediction includes encoding the test texts

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
start = time.time()
E_train = embedder.encode(..., prompt=INSTRUCTION, batch_size=64)
classifier = LogisticRegression(max_iter=2000, C=10).fit(..., ...)
trained = time.time() - start
start = time.time()
E_test = embedder.encode(..., prompt=INSTRUCTION, batch_size=64)
predictions = classifier.predict(...)
record("Embeddings + regression", predictions, trained, time.time() - start)
```

### Solution

In [ ]:
start = time.time()
E_train = embedder.encode(train_part.text.tolist(), prompt=INSTRUCTION, batch_size=64)
classifier = LogisticRegression(max_iter=2000, C=10).fit(E_train, train_part.label)
trained = time.time() - start
start = time.time()
E_test = embedder.encode(test_part.text.tolist(), prompt=INSTRUCTION, batch_size=64)
predictions = classifier.predict(E_test)
record("Embeddings + regression", predictions, trained, time.time() - start)

## 3. Fine-Tune an Encoder

Fine-tune `ENCODER` (a pretrained encoder, plus a classification layer) on the training data, with `transformers`: tokenize the texts, number the intents, and train with a `Trainer`.

Training takes a few minutes: start it, and meanwhile read the results of steps 1 and 2 again. Then predict the test intents (`trainer.predict`, on `test_part`) and record the result.

Without a GPU, the full training would take hours: `MAX_STEPS` stops it after 10 steps, to see the loop run, and the prediction covers 64 texts only. The next exercise then loads the test logits of the model trained on a GPU, which is what the rest of the lab needs (the model itself, 600 MB per language, isn't published). Their macro F1 is 0.869; a GPU run's may differ from it by a point or so (0.86 to 0.87 in ours): GPU computations don't reproduce exactly from one card or library version to another.

**Useful documentation**

- [Fine-tuning a pretrained model](https://huggingface.co/docs/transformers/training)
- [`Dataset.from_dict`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.from_dict)
- [`Dataset.map`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.map)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`AutoModelForSequenceClassification`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification)
- [`TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)
- [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer)
- [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorWithPadding)
- [`Trainer.predict`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.predict)

In [ ]:
# Without a GPU, the previous exercise encoded 64 texts of each only: we load the vectors
# of all of them, computed on a GPU by the same code (stored in float16), redo the
# regression, and record its result with the timings measured on a GPU.
if not GPU:
  E_train = np.load("massive-intents/lab-results/embeddings-train-en.npy").astype(
    np.float32
  )
  E_test = np.load("massive-intents/lab-results/embeddings-test-en.npy").astype(
    np.float32
  )
  classifier = LogisticRegression(max_iter=2000, C=10).fit(E_train, train.label)
  gpu = pd.read_csv("massive-intents/lab-results/timings-en.csv").set_index("method")
  name = "Embeddings + regression"
  predict_seconds = gpu.ms_per_text[name] * len(test) / 1000
  record(name, classifier.predict(E_test), gpu.train_s[name], predict_seconds)

In [ ]:
# Multilingual, 2025; on par with ModernBERT on this data
ENCODER = "jhu-clsp/mmBERT-base"
# For CamemBERTav2, AutoTokenizer (transformers 5) splits the text into characters:
# about 0.3 macro F1. With the tokenizer from its tokenizer.json, 0.68 to 0.72.
# 3 epochs on a GPU (a few minutes); without one, 10 steps (about two minutes).
MAX_STEPS = -1 if GPU else 10

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- numbering the intents: `index = {label: i for i, label in enumerate(labels)}`
- `tokenizer(batch["text"], truncation=True, max_length=64)`, applied by `Dataset.map(..., batched=True, remove_columns=["text"])`: the function gets batches of texts, and the `Trainer` only wants numeric columns
- `Dataset.from_dict({"text": ..., "label": ...})`: a dataset from lists
- `AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=len(labels))`
- `TrainingArguments("encoder", num_train_epochs=3, learning_rate=5e-5, per_device_train_batch_size=32, fp16=GPU, max_steps=MAX_STEPS, warmup_steps=0.1, save_strategy="no", report_to="none")`; `fp16=GPU`: half precision, on a GPU only; `max_steps=-1`: no limit, the 3 epochs; `warmup_steps=0.1`: a number below 1 is a share of the training steps
- `Trainer(model=..., args=..., train_dataset=..., data_collator=DataCollatorWithPadding(tokenizer))`: the collator pads each batch to the length of its longest text
- `trainer.predict(to_dataset(test_part)).predictions`: the logits, one row per text; `argmax(axis=1)` gives the predicted class, to convert into an intent with `labels[i]`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from datasets import Dataset
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
)

index = {label: i for i, label in enumerate(labels)}
tokenizer = AutoTokenizer.from_pretrained(ENCODER)


def to_dataset(df: pd.DataFrame) -> Dataset:
  ds = Dataset.from_dict(
    {"text": df.text.tolist(), "label": [... for label in df.label]}
  )  # each intent's number
  return ds.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=64),
    batched=True,
    remove_columns=[...],
  )


model = AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=...)
args = TrainingArguments(
  "encoder",
  num_train_epochs=3,
  learning_rate=5e-5,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=128,
  fp16=GPU,
  max_steps=MAX_STEPS,
  warmup_steps=0.1,
  logging_steps=100,
  save_strategy="no",
  report_to="none",
)
trainer = Trainer(
  model=model,
  args=args,
  train_dataset=...,
  data_collator=DataCollatorWithPadding(tokenizer),
)
start = time.time()
trainer.train()
trained = time.time() - start

start = time.time()
logits = trainer.predict(...).predictions
predictions = [labels[i] for i in ...]  # each text's predicted class
record("Fine-tuned encoder", predictions, trained, time.time() - start)
```

### Solution

In [ ]:
from datasets import Dataset
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
)

index = {label: i for i, label in enumerate(labels)}
tokenizer = AutoTokenizer.from_pretrained(ENCODER)


def to_dataset(df: pd.DataFrame) -> Dataset:
  ds = Dataset.from_dict(
    {"text": df.text.tolist(), "label": [index[label] for label in df.label]}
  )
  return ds.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=64),
    batched=True,
    remove_columns=["text"],
  )


model = AutoModelForSequenceClassification.from_pretrained(
  ENCODER, num_labels=len(labels)
)
args = TrainingArguments(
  "encoder",
  num_train_epochs=3,
  learning_rate=5e-5,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=128,
  fp16=GPU,
  max_steps=MAX_STEPS,
  warmup_steps=0.1,
  logging_steps=100,
  save_strategy="no",
  report_to="none",
)
trainer = Trainer(
  model=model,
  args=args,
  train_dataset=to_dataset(train),
  data_collator=DataCollatorWithPadding(tokenizer),
)
start = time.time()
trainer.train()
trained = time.time() - start

start = time.time()
logits = trainer.predict(to_dataset(test_part)).predictions
predictions = [labels[i] for i in logits.argmax(axis=1)]
record("Fine-tuned encoder", predictions, trained, time.time() - start)

## 4. Keep the Results, and Read the F1 Closely

Tomorrow, the next lab compares these methods with an LLM. The next cell saves each method's predictions to `predictions-en.csv` (one row per test text: `id`, `label`, then one column per method) and the timings to `timings-en.csv`. Run it, then download both files (*Files* panel on the left, or `google.colab.files.download`): the next lab loads them. A copy of our results is published, just in case.

In [ ]:
# Without a GPU, the encoder trained for 10 steps only: we load the test logits of the model
# trained on a GPU by the same code, and record its result with the timings measured on a
# GPU.
if not GPU:
  logits = np.load("massive-intents/lab-results/encoder-logits-en.npy")
  gpu = pd.read_csv("massive-intents/lab-results/timings-en.csv").set_index("method")
  name = "Fine-tuned encoder"
  predictions = [labels[i] for i in logits.argmax(axis=1)]
  predict_seconds = gpu.ms_per_text[name] * len(test) / 1000
  record(name, predictions, gpu.train_s[name], predict_seconds)

# Each method's predictions and timings, for the next lab
predictions = pd.DataFrame({"id": test.id, "label": test.label})
for name, result in results.items():
  predictions[name] = result["predictions"]
predictions.to_csv("predictions-en.csv", index=False)
timings = pd.DataFrame(
  [(name, r["train_s"], r["ms_per_text"]) for name, r in results.items()],
  columns=["method", "train_s", "ms_per_text"],
)
timings.to_csv("timings-en.csv", index=False)
timings

### What the Macro F1 Hides

A macro F1 sums up 60 intents in one number. With `classification_report`, list the 5 intents with the fine-tuned encoder's lowest F1, and compare them with their number of test texts (the *support*) and their confusions (`confusions`): which ones are real weaknesses? Then count the test texts already seen in training (lowercased, without leading or trailing spaces), and compute again, without them, the macro F1 of each method in `results`: what were we measuring?

**Useful documentation**

- [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `encoder = results["Fine-tuned encoder"]["predictions"]`: the encoder's predictions, one per test text
- `classification_report(test.label, encoder, output_dict=True, zero_division=0)`: a dictionary, one entry per intent (`precision`, `recall`, `f1-score`, `support`) plus `accuracy` and two averages; `zero_division=0` counts 0, without a warning, for an intent never predicted or with no test text
- `pd.DataFrame(report).T`: one row per intent; `.drop(["accuracy", "macro avg", "weighted avg"])` removes the others, `.sort_values("f1-score").head(5)` keeps the 5 lowest
- `confusions(encoder, n=None)`: all the confused (true, predicted) pairs, from the most to the least frequent; keep those where the intent appears
- `test.text.str.lower().str.strip()`: the texts lowercased, without leading or trailing spaces; `.isin(set(...))` says, for each test text, whether it is in the training set
- `f1_score(truth, predictions[~seen], average="macro", labels=truth.unique())`, with `truth = test.label[~seen]`, and `seen` and `predictions` as `numpy` arrays: the macro F1 without the seen texts (`~` inverts a boolean mask), over these texts' intents, as in `record`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from sklearn.metrics import classification_report

encoder = results["Fine-tuned encoder"]["predictions"]
report = classification_report(test.label, encoder, output_dict=True, zero_division=0)
scores = pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"])
weakest = scores.sort_values(...).head(5)
print(weakest.round(2))
for intent in weakest.index:
  pairs = [(pair, count) for pair, count in confusions(encoder, n=None) if ...]
  print(intent, pairs[:3])  # its three most frequent confusions


def normalize(texts: pd.Series) -> pd.Series:
  return ...  # lowercased, without leading or trailing spaces


seen = normalize(test.text).isin(set(normalize(...))).to_numpy()
print(seen.sum(), "test texts out of", len(test), "already seen in training")
for name, result in results.items():
  predictions = np.array(result["predictions"])
  truth = test.label[...]
  f1 = f1_score(truth, predictions[...], average="macro", labels=truth.unique())
  print(f"{name:30} macro F1 without them {f1:.3f}")
```

#### Solution

In [ ]:
from sklearn.metrics import classification_report

encoder = results["Fine-tuned encoder"]["predictions"]
report = classification_report(test.label, encoder, output_dict=True, zero_division=0)
scores = pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"])
weakest = scores.sort_values("f1-score").head(5)
print(weakest.round(2))
for intent in weakest.index:
  pairs = [
    (pair, count) for pair, count in confusions(encoder, n=None) if intent in pair
  ]
  print(intent, pairs[:3])


def normalize(texts: pd.Series) -> pd.Series:
  return texts.str.lower().str.strip()


seen = normalize(test.text).isin(set(normalize(train.text))).to_numpy()
print(seen.sum(), "test texts out of", len(test), "already seen in training")
for name, result in results.items():
  predictions = np.array(result["predictions"])
  truth = test.label[~seen]
  f1 = f1_score(truth, predictions[~seen], average="macro", labels=truth.unique())
  print(f"{name:30} macro F1 without them {f1:.3f}")

# With our published logits (without a GPU); those of an encoder trained on a GPU
# give similar figures. The lowest: music_settings (6 test texts, confused with
# play_music), iot_hue_lighton (3 texts: an F1 on so few says almost nothing),
# email_addcontact (12 texts, confused with email_sendemail), general_quirky (169
# texts, F1 0.65) and music_dislikeness (4 texts). Only general_quirky is a real
# weakness: the catch-all of the requests with no precise intent, confused with
# news_query and qa_factoid.
# 21 test texts out of 2,974 are already in the training set ("quiet", "what's the
# weather like tomorrow"): without them, the encoder's F1 goes from 0.869 to 0.868,
# the TF-IDF regression's from 0.798 to 0.797. Nothing changes here; in French, 150
# texts are, and every method loses half a point to almost two points: the test then
# measured some memory on top of generalization.

## 5. Bonus

### SetFit

With only 8 examples per intent, fine-tune an *embedding* model contrastively, then a classification head ([SetFit](https://github.com/huggingface/setfit)). Where does it land, with about 24 times fewer labels? Without a GPU, `max_steps=MAX_STEPS` stops the contrastive learning after 10 steps, and the prediction covers `test_part`: a few minutes instead of about three hours, for a worse result; the F1 of the full training, on a GPU, is in the solution.

**Useful documentation**

- [SetFit: quickstart](https://huggingface.co/docs/setfit/quickstart)
- [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)
- [`DataFrameGroupBy.head`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.head.html)
- [`setfit.TrainingArguments`](https://huggingface.co/docs/setfit/reference/trainer#setfit.TrainingArguments)
- [`multilingual-e5-base`](https://huggingface.co/intfloat/multilingual-e5-base)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `train.sample(frac=1, random_state=0).groupby("label").head(8)`: shuffles the examples, then keeps the first 8 of each intent (one of them only has 4)
- `SetFitModel.from_pretrained("intfloat/multilingual-e5-base", labels=labels)`: a good starting model, whose texts must start with `"query: "`
- `setfit.TrainingArguments(batch_size=32, num_epochs=1, num_iterations=10, max_steps=MAX_STEPS, report_to="none")`: `num_iterations` is the number of pairs generated per example for contrastive learning; `max_steps=MAX_STEPS`: no limit on a GPU, 10 steps without one (exercise 3)
- `setfit.Trainer(model=..., args=..., train_dataset=Dataset.from_dict({"text": ..., "label": ...}))`, then `.train()`; `model.predict(list_of_texts)` to predict
- `%pip install -q setfit` as the cell's first line; import `Trainer` and `TrainingArguments` from `setfit` under other names (`SetFitTrainer`, `SetFitArguments`) so as not to overwrite those of `transformers`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import (
  SetFitModel,
  Trainer as SetFitTrainer,
  TrainingArguments as SetFitArguments,
)

few = train.sample(frac=1, random_state=0).groupby("label").head(...)
setfit_model = SetFitModel.from_pretrained(
  "intfloat/multilingual-e5-base", labels=labels
)
setfit_trainer = SetFitTrainer(
  model=setfit_model,
  args=SetFitArguments(
    batch_size=32,
    num_epochs=1,
    num_iterations=10,
    max_steps=MAX_STEPS,
    report_to="none",
  ),
  train_dataset=HFDataset.from_dict(
    {"text": ("query: " + few.text).tolist(), "label": ...}
  ),
)
start = time.time()
setfit_trainer.train()
trained = time.time() - start
start = time.time()
predictions = setfit_model.predict(...)  # the test_part texts, preceded by "query: "
record("SetFit (8 per class)", list(predictions), trained, time.time() - start)
```

#### Solution

In [ ]:
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import (
  SetFitModel,
  Trainer as SetFitTrainer,
  TrainingArguments as SetFitArguments,
)

few = train.sample(frac=1, random_state=0).groupby("label").head(8)
setfit_model = SetFitModel.from_pretrained(
  "intfloat/multilingual-e5-base", labels=labels
)
setfit_trainer = SetFitTrainer(
  model=setfit_model,
  args=SetFitArguments(
    batch_size=32,
    num_epochs=1,
    num_iterations=10,
    max_steps=MAX_STEPS,
    report_to="none",
  ),
  train_dataset=HFDataset.from_dict(
    {"text": ("query: " + few.text).tolist(), "label": few.label.tolist()}
  ),
)
start = time.time()
setfit_trainer.train()
trained = time.time() - start
start = time.time()
predictions = setfit_model.predict(("query: " + test_part.text).tolist())
record("SetFit (8 per class)", list(predictions), trained, time.time() - start)

# On a GPU, about a minute of training, and a macro F1 of 0.669 on the whole test:
# with 8 examples per intent, SetFit stays far below the encoder fine-tuned on the
# whole training set (0.86 to 0.87), and even below TF-IDF + Naive Bayes (0.719).
# Without a GPU, 10 steps and 64 test texts say almost nothing about its worth.

### Calibration

For the regression on *embeddings* and the fine-tuned encoder, plot the reliability diagram: the share of correct predictions as a function of the announced probability (`sklearn.calibration.calibration_curve`, on "is the prediction correct?" and the predicted class's probability). Which one is better calibrated?

**Useful documentation**

- [`calibration_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html)
- [`LogisticRegression.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
- [`scipy.special.softmax`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `calibration_curve(correct, confidence, n_bins=10, strategy="quantile")`: `correct` says whether each prediction is right (booleans), `confidence` is the announced probability; `strategy="quantile"` puts as many predictions in each bin; returns each bin's share of correct predictions and mean confidence
- `classifier.predict_proba(E_test)`: one row per text, the columns follow the order of `classifier.classes_`
- `softmax(logits, axis=1)`: the fine-tuned encoder's logits, turned into probabilities; its columns follow the order of `labels`
- `proba.max(axis=1)`: the predicted class's probability; `proba.argmax(axis=1)`: its number

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilities = {
  "Embeddings + regression": classifier.predict_proba(E_test),
  "Fine-tuned encoder": softmax(..., axis=1),
}
for name, proba in probabilities.items():
  classes = classifier.classes_ if name.startswith("Embeddings") else np.array(labels)
  confidence = proba.max(axis=1)
  correct = classes[proba.argmax(axis=1)] == ...  # the test's true intents
  print(f"{name:25} confidence {confidence.mean():.2f}  correct {correct.mean():.3f}")
  accuracy, mean_confidence = calibration_curve(
    ..., ..., n_bins=10, strategy="quantile"
  )
  plt.plot(mean_confidence, accuracy, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Announced probability")
plt.ylabel("Share of correct predictions")
plt.legend()
plt.show()
```

#### Solution

In [ ]:
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilities = {
  "Embeddings + regression": classifier.predict_proba(E_test),
  "Fine-tuned encoder": softmax(logits, axis=1),
}
for name, proba in probabilities.items():
  classes = classifier.classes_ if name.startswith("Embeddings") else np.array(labels)
  confidence = proba.max(axis=1)
  correct = classes[proba.argmax(axis=1)] == test.label.to_numpy()
  print(f"{name:25} confidence {confidence.mean():.2f}  correct {correct.mean():.3f}")
  accuracy, mean_confidence = calibration_curve(
    correct, confidence, n_bins=10, strategy="quantile"
  )
  plt.plot(mean_confidence, accuracy, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Announced probability")
plt.ylabel("Share of correct predictions")
plt.legend()
plt.show()

# With our published vectors and logits (without a GPU); on a GPU, similar figures.
# The regression on embeddings is the better calibrated: mean confidence 0.86 for 88%
# correct predictions, and its curve follows the diagonal, slightly above it (it
# slightly underestimates itself). The fine-tuned encoder, more accurate, is
# overconfident: mean confidence 0.96 for 88.6% correct, and its curve runs below the
# diagonal. Three quarters of its predictions announce more than 0.99, and only 96% of
# those are correct: before deciding on its probability (a threshold, a hand-off to a
# human), it has to be recalibrated on a validation set.